## Inferential Statistic for Laptop Price

    In this page will have been made paired hypothesis test for Actual_Price and Dicount_Price. Then will have been established a regression model and infered how much bought a desktop. Also, therefor will have been look at looked at independent categorical variables do have an impress.

In [1]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sbn

In [7]:
df = pd.read_pickle("master_laptop_data.pkl")

df.head()

,Laptop_ID,Discounted_Price,Actual_Price,Rating,Reviews,Brand,Core,SSD,Series,Processor,RAM,Storage
0,0,2034990.0,2580000.0,5.0,2.0,Apple,M1,NaN,MacBook,Apple M1,8GB,256GB SSD
1,1,2819990.0,3500000.0,43.0,5.0,Apple,M4,NaN,MacBook,NaN,NaN,NaN
2,2,2859990.0,3300000.0,NaN,NaN,Asus,Ultra 7,512GB,Zenbook,Intel Core Ultra 7,16GB,512GB SSD
3,3,2879990.0,3400000.0,NaN,NaN,Lenovo,Ultra 7,NaN,ThinkPad,Intel Core Ultra 7,16GB,512GB SSD
4,4,1469990.0,1750000.0,NaN,NaN,Lenovo,Ryzen 7,NaN,IdeaPad,AMD Ryzen 7,8GB,512GB SSD


## Plan
First, it will be tested whether the observations across all numeric variables are paired in terms of their means and variances. Then, confidence intervals and point estimates will be calculated separately. Finally, based on the results of the hypothesis tests, a regression model will be developed to conclude the study.

In [8]:
from scipy.stats import mannwhitneyu, wilcoxon
import pandas as pd

# 1. Data Type Conversion and Missing Value Handling
# Create a clean copy focusing only on the two variables for analysis
test_df = df[['Actual_Price', 'Discounted_Price']].dropna()

# Ensure values are numeric (float); 'coerce' turns invalid parsing into NaN
test_df['Actual_Price'] = pd.to_numeric(test_df['Actual_Price'], errors='coerce')
test_df['Discounted_Price'] = pd.to_numeric(test_df['Discounted_Price'], errors='coerce')

# Drop any new NaNs created during numeric conversion
test_df = test_df.dropna()

print(f"Number of rows included in the analysis: {len(test_df)}")

if len(test_df) > 0:
    # 2. Re-running the Non-Parametric Hypothesis Tests
    # Wilcoxon: Compares medians of paired samples (Before vs After)
    stat_wil, p_wil = wilcoxon(test_df['Actual_Price'], test_df['Discounted_Price'])
    
    # Mann-Whitney U: Compares the distributions of two independent-like samples
    stat_mw, p_mw = mannwhitneyu(test_df['Actual_Price'], test_df['Discounted_Price'])

    print(f"\n{'='*30}")
    print("HYPOTHESIS TEST RESULTS (CLEANED)")
    print(f"{'='*30}")

    print(f"\n[Wilcoxon Signed-Rank Test]")
    print(f"P-value: {p_wil:.4e}")

    print(f"\n[Mann-Whitney U Test]")
    print(f"P-value: {p_mw:.4e}")
else:
    print("ERROR: No data remaining after cleaning. Please check column names and data content.")

Number of rows included in the analysis: 291

HYPOTHESIS TEST RESULTS (CLEANED)

[Wilcoxon Signed-Rank Test]
P-value: 1.7917e-49

[Mann-Whitney U Test]
P-value: 9.2202e-06


## Interpretation:
The Wilcoxon Signed-Rank test indicates that there is a statistically significant difference between the actual price and the discounted price, as expected. The Mann–Whitney U test also shows that the distributions of these two price variables differ in terms of their central tendency. In addition, the correlation analysis reveals a strong and statistically significant relationship between these two variables. Therefore, there is both a systematic difference between the variables and a strong relationship between them.


In [9]:
import itertools # Crucial for generating combinations
from scipy.stats import mannwhitneyu, wilcoxon
import pandas as pd

# List of numerical columns for comparison
test_cols = ["Reviews", "Rating", "Discounted_Price", "Actual_Price"]
comparison_results = []

# Generate all unique pairs from the list
pairs = list(itertools.combinations(test_cols, 2))

print(f"{'='*40}\nNON-PARAMETRIC DISTRIBUTION TESTS\n{'='*40}")

for col1, col2 in pairs:
    # Skip the pair already analyzed
    if (col1 == "Actual_Price" and col2 == "Discounted_Price") or (col1 == "Discounted_Price" and col2 == "Actual_Price"):
        continue

    # Prepare cleaned and numeric data for the pair
    temp_df = df[[col1, col2]].dropna()
    temp_df[col1] = pd.to_numeric(temp_df[col1], errors='coerce')
    temp_df[col2] = pd.to_numeric(temp_df[col2], errors='coerce')
    temp_df = temp_df.dropna()

    if not temp_df.empty:
        # Independent distribution comparison
        stat_mw, p_mw = mannwhitneyu(temp_df[col1], temp_df[col2])
        
        # Paired/Related distribution comparison
        try:
            stat_wil, p_wil = wilcoxon(temp_df[col1], temp_df[col2])
        except Exception:
            p_wil = "Error"

        comparison_results.append({
            "Variable 1": col1,
            "Variable 2": col2,
            "MW-U p-value": f"{p_mw:.4e}",
            "Wilcoxon p-value": f"{p_wil:.4e}" if isinstance(p_wil, float) else p_wil
        })

# Store and display results
comp_df = pd.DataFrame(comparison_results)
display(comp_df)

NON-PARAMETRIC DISTRIBUTION TESTS


,Variable 1,Variable 2,MW-U p-value,Wilcoxon p-value
0,Reviews,Rating,6.7740e-10,5.4626e-06
1,Reviews,Discounted_Price,2.7819e-10,8.2842e-06
2,Reviews,Actual_Price,2.7819e-10,8.2842e-06
3,Rating,Discounted_Price,2.0122e-10,8.2842e-06
4,Rating,Actual_Price,2.0122e-10,8.2842e-06


## Interpretation:
There are statistically significant distributional and positional differences among the variables Reviews, Rating, Actual_Price, and Discounted_Price. Additionally, as seen in previous correlation analyses, the Rating and Reviews variables do not affect either each other or the price variables. Furthermore, the lack of dispersion and the highly uniform nature of their distributions suggest that these variables are generally not meaningful in an explanatory sense.


## Coinfidence Interval and Point Estimates

Due to the lack of meaningful results for the Reviews and Rating variables in the tests performed above and in the previous script, confidence intervals and point estimates will only be computed for the Actual Price and Discounted Price variables in this section.
The variables are not normally distributed; therefore, bootstrap methods will be used.


In [10]:
def bootstrap_ci(data, iterations=10000, alpha=0.05):
    """
    Calculates the Bootstrap Confidence Interval for the mean.
    """
    # Clear NaNs
    clean_data = data.dropna().values
    boot_means = []
    
    # Resampling process
    for _ in range(iterations):
        sample = np.random.choice(clean_data, size=len(clean_data), replace=True)
        boot_means.append(np.mean(sample))
    
    # Calculate Percentile CI (95%)
    lower = np.percentile(boot_means, (alpha/2) * 100)
    upper = np.percentile(boot_means, (1 - alpha/2) * 100)
    point_estimate = np.mean(clean_data)
    
    return point_estimate, lower, upper

# Execution for both variables
print(f"{'='*45}\nBOOTSTRAP CONFIDENCE INTERVALS (N=10000)\n{'='*45}")

for col in ['Actual_Price', 'Discounted_Price']:
    point, low, high = bootstrap_ci(df[col])
    print(f"\n--- {col} ---")
    print(f"Point Estimate (Mean): {point:.2f}")
    print(f"Bootstrap 95% CI: [{low:.2f}, {high:.2f}]")

BOOTSTRAP CONFIDENCE INTERVALS (N=10000)

--- Actual_Price ---
Point Estimate (Mean): 3212615.88
Bootstrap 95% CI: [3016084.19, 3428905.83]

--- Discounted_Price ---
Point Estimate (Mean): 2775878.92
Bootstrap 95% CI: [2598137.84, 2980879.84]


## Interpretation:
At the 95% confidence level, both variables’ point estimates and confidence intervals were calculated using the bootstrap method. Accordingly, there is an approximate difference of 500,000 between the actual price and the discounted price in terms of both the point estimates and their confidence intervals.